# BTP — A Cell That Decides, Moves, and Resets

## Decide → deform → move → relax → decide again

Every version so far hands the cell a polarization once, at $t=0$,
and lets it crawl in whatever direction that implies for the rest of
the run. This notebook adds a simple **decision layer** on top of the
exact Version 3 mechanics (full Stokes flow, active/elastic/Frank
stress, substrate friction, anchoring) — the physics is copied over
unchanged; what's new is the control policy driving it:

1. **Decide.** Pick a random direction $\theta\sim U(0,2\pi)$ and a
   random travel distance.
2. **Deform and move.** Point the polarization $\mathbf P$ along
   $\theta$ (active stress $\zeta$ and anchoring $\beta$ switched on,
   exactly the "lamellipodium" morphology from Version 3's Section 14,
   just reoriented). The cell polarises, deforms into its motile fan
   shape, and crawls in that direction.
3. **Relax.** Once it has travelled the chosen distance, switch the
   activity off ($\zeta=\beta=w_0=0$). With no energy being pumped in
   to sustain the fan, interfacial tension — the same passive physics
   validated in Version 3's Section 13 — pulls the cell back towards
   a circle. We wait either for a fixed minimum time or until the
   circularity comes back above a target, whichever is reached last.
4. **Repeat.** Pick a new random direction and distance, and go again.

This is a minimal model of **stop-and-go / repolarising migration**:
the cell is not steered externally and does not remember its previous
heading — each leg starts from the relaxed, roughly circular shape
and a fresh coin flip for direction.


# 1. The Version 3 physics, unchanged

Grid, derivatives, thermodynamics, body force, Stokes solve, initial condition and diagnostics — copied from `BTP_version3.ipynb` with no changes. If you already have that notebook open, every cell in this section should look familiar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import Video


# ---------------- grid ----------------
N = 128
L = 80.0

# ---------------- cell ----------------
R0 = 13.0
eps = 1.5
a_phi = 1.0
M_phi = 1.0

# ---------------- polarization ----------------
a_pol = 0.5
kappa = 1.0
Gamma = 1.0
chi = 1.1

# ---------------- activity (used during the "move" phase) ----------------
w0 = 1.0
zeta = -1.0
beta = 0.25

# ---------------- substrate and fluid ----------------
eta = 1.0
xi = 1.0

# ---------------- time ----------------
dt = 0.01
sample_every = 100

print(f"grid {N} x {N},  dx = {L/N:.4f}")
print(f"stability: dt must be < dx^2/(4 M eps^2) = "
      f"{(L/N)**2/(4*M_phi*eps**2):.4f};  using dt = {dt}")


In [ ]:
def setup_grid(N_=N, L_=L):
    global N, L, dx, X, Y, KX, KY, K2, K2_safe
    N, L = N_, L_
    dx = L / N
    coords = (np.arange(N) - N // 2) * dx
    X, Y = np.meshgrid(coords, coords, indexing="ij")
    k1 = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
    KX, KY = np.meshgrid(k1, k1, indexing="ij")
    K2 = KX**2 + KY**2
    K2_safe = K2.copy()
    K2_safe[0, 0] = 1.0


setup_grid()


def d_dx(a):
    return (np.roll(a, -1, axis=0) - np.roll(a, 1, axis=0)) / (2.0 * dx)


def d_dy(a):
    return (np.roll(a, -1, axis=1) - np.roll(a, 1, axis=1)) / (2.0 * dx)


def laplacian(a):
    return (np.roll(a, -1, axis=0) + np.roll(a, 1, axis=0)
            + np.roll(a, -1, axis=1) + np.roll(a, 1, axis=1)
            - 4.0 * a) / dx**2


def divergence_of_flux(phi, ux, uy):
    Fx = 0.5 * (phi + np.roll(phi, -1, axis=0)) * 0.5 * (ux + np.roll(ux, -1, axis=0))
    Fy = 0.5 * (phi + np.roll(phi, -1, axis=1)) * 0.5 * (uy + np.roll(uy, -1, axis=1))
    return ((Fx - np.roll(Fx, 1, axis=0)) + (Fy - np.roll(Fy, 1, axis=1))) / dx


In [ ]:
def chemical_potential_phi(phi, P2, dPx_dx, dPy_dy):
    return (2.0 * a_phi * phi * (phi - 1.0) * (2.0 * phi - 1.0)
            - eps**2 * laplacian(phi)
            - a_pol * P2
            - beta_run * (dPx_dx + dPy_dy))


def molecular_field_P(phi, Px, Py, P2):
    bulk = -a_pol * (2.0 * phi - 1.0) + a_pol * P2
    hx = bulk * Px - kappa_run * laplacian(Px) + beta_run * d_dx(phi)
    hy = bulk * Py - kappa_run * laplacian(Py) + beta_run * d_dy(phi)
    return hx, hy


def total_free_energy(phi, Px, Py, P2):
    gx, gy = d_dx(phi), d_dy(phi)
    Pxx, Pxy = d_dx(Px), d_dy(Px)
    Pyx, Pyy = d_dx(Py), d_dy(Py)
    density = (a_phi * phi**2 * (phi - 1.0)**2
               + 0.5 * eps**2 * (gx**2 + gy**2)
               - 0.5 * a_pol * (2.0 * phi - 1.0) * P2
               + 0.25 * a_pol * P2**2
               + 0.5 * kappa_run * (Pxx**2 + Pxy**2 + Pyx**2 + Pyy**2)
               + beta_run * (Px * gx + Py * gy))
    return density.sum() * dx**2


In [ ]:
def body_force(phi, Px, Py, mu, hx, hy):
    # ---------- active stress:  -zeta phi P_a P_b ----------
    Axx = -zeta_run * phi * Px * Px
    Axy = -zeta_run * phi * Px * Py
    Ayy = -zeta_run * phi * Py * Py

    # ---------- elastic stress ----------
    torque = 0.5 * (Px * hy - Py * hx)
    Exx = -chi_run * Px * hx
    Exy = torque - 0.5 * chi_run * (Px * hy + Py * hx)
    Eyx = -torque - 0.5 * chi_run * (Py * hx + Px * hy)
    Eyy = -chi_run * Py * hy

    # ---------- Frank stress:  -kappa d_a P_g d_b P_g ----------
    Pxx, Pxy = d_dx(Px), d_dy(Px)
    Pyx, Pyy = d_dx(Py), d_dy(Py)
    Kxx = -kappa_run * (Pxx * Pxx + Pyx * Pyx)
    Kxy = -kappa_run * (Pxx * Pxy + Pyx * Pyy)
    Kyy = -kappa_run * (Pxy * Pxy + Pyy * Pyy)

    # ---------- total, then f_a = d_b sigma_ab ----------
    Sxx = Axx + Exx + Kxx
    Sxy = Axy + Exy + Kxy
    Syx = Axy + Eyx + Kxy
    Syy = Ayy + Eyy + Kyy

    fx = d_dx(Sxx) + d_dy(Sxy) - phi * d_dx(mu)
    fy = d_dx(Syx) + d_dy(Syy) - phi * d_dy(mu)
    return fx, fy


def stokes_solve(fx, fy):
    fx_h = np.fft.fft2(fx)
    fy_h = np.fft.fft2(fy)
    longitudinal = (KX * fx_h + KY * fy_h) / K2_safe
    fx_h = fx_h - KX * longitudinal
    fy_h = fy_h - KY * longitudinal
    screened = xi_run + eta * K2
    vx_h = fx_h / screened
    vy_h = fy_h / screened
    vx_h[0, 0] = 0.0
    vy_h[0, 0] = 0.0
    return np.real(np.fft.ifft2(vx_h)), np.real(np.fft.ifft2(vy_h))


In [ ]:
def initialize(R0=R0, eps=eps, ecc=0.0, P_init="uniform", noise=0.02, seed=7):
    radius = np.sqrt((X / (1.0 + ecc))**2 + (Y * (1.0 + ecc))**2)
    phi = 0.5 * (1.0 - np.tanh((radius - R0) / (np.sqrt(2.0) * eps)))
    rng = np.random.default_rng(seed)
    if P_init == "uniform":
        Px = phi * (1.0 + noise * rng.standard_normal(phi.shape))
        Py = phi * (noise * rng.standard_normal(phi.shape))
    elif P_init == "radial":
        r = np.sqrt(X**2 + Y**2) + 1e-9
        Px, Py = phi * X / r, phi * Y / r
    else:  # "zero"
        Px = phi * noise * rng.standard_normal(phi.shape)
        Py = phi * noise * rng.standard_normal(phi.shape)
    magnitude = np.sqrt(Px**2 + Py**2) + 1e-12
    Px = phi * Px / magnitude
    Py = phi * Py / magnitude
    return phi, Px, Py


def centre_of_mass(phi):
    mass = phi.sum()
    return (X * phi).sum() / mass, (Y * phi).sum() / mass


def touches_boundary(phi, margin=6):
    inside = phi > 0.5
    return bool(inside[:margin].any() or inside[-margin:].any()
                or inside[:, :margin].any() or inside[:, -margin:].any())


def calculate_diagnostics(phi, Px, Py, vx, vy, P2):
    mass = phi.sum()
    inside = phi > 0.5
    area = inside.sum() * dx**2
    gx, gy = d_dx(phi), d_dy(phi)
    perimeter = np.sqrt(gx**2 + gy**2).sum() * dx**2
    circularity = 4.0 * np.pi * area / max(perimeter**2, 1e-12)

    xc = (X * phi).sum() / mass
    yc = (Y * phi).sum() / mass
    Xc, Yc = X - xc, Y - yc
    Mxx = (Xc**2 * phi).sum() / mass
    Myy = (Yc**2 * phi).sum() / mass
    Mxy = (Xc * Yc * phi).sum() / mass
    ev = np.linalg.eigvalsh(np.array([[Mxx, Mxy], [Mxy, Myy]]))
    minor = np.sqrt(max(ev[0], 1e-12))
    major = np.sqrt(max(ev[1], 1e-12))

    order = np.hypot((Px * phi).sum() / mass, (Py * phi).sum() / mass)
    return {
        "area": area, "circularity": circularity,
        "aspect": major / minor,
        "P_order": order,
        "v_max": np.sqrt(vx**2 + vy**2).max(),
    }


def draw_cell(ax, phi, Ax, Ay, direction=None, stride=6, title=None, arrow_scale=None):
    ax.imshow(phi.T, origin="lower", vmin=0.0, vmax=1.0,
              extent=[-L / 2, L / 2, -L / 2, L / 2])
    ax.contour(X, Y, phi, [0.5], colors="white", linewidths=0.8)
    if arrow_scale is None:
        arrow_scale = 14.0 * (np.hypot(Ax, Ay).max() + 1e-12)
    ax.quiver(X[::stride, ::stride], Y[::stride, ::stride],
              Ax[::stride, ::stride], Ay[::stride, ::stride],
              color="white", scale=arrow_scale, width=0.004)
    if direction is not None:
        Vx, Vy = direction
        n = np.hypot(Vx, Vy)
        if n > 1e-6:
            ax.arrow(-0.42 * L, -0.42 * L, 0.17 * L * Vx / n, 0.17 * L * Vy / n,
                     color="orangered", width=0.017 * L, head_width=0.05 * L)
    ax.set_xlim(-L / 2, L / 2)
    ax.set_ylim(-L / 2, L / 2)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    if title:
        ax.set_title(title, fontsize=9)


# 2. The decision layer

`advance_one_step` is exactly steps A–E of Version 3's time loop
(`run_simulation`'s inner body), pulled out into its own function so
the new driver below can call it once per step regardless of which
phase the cell is in. The only thing that changes between phases is
which global parameters (`zeta_run`, `beta_run`) and which `w0` are
passed in — the equations themselves are untouched.


In [ ]:
def advance_one_step(phi, Px, Py, vx, vy, w0_now, dt=dt, flow=True):
    P2 = Px**2 + Py**2
    mu = chemical_potential_phi(phi, P2, d_dx(Px), d_dy(Py))
    hx, hy = molecular_field_P(phi, Px, Py, P2)

    if flow:
        fx, fy = body_force(phi, Px, Py, mu, hx, hy)
        vx, vy = stokes_solve(fx, fy)

    ux = vx + w0_now * Px
    uy = vy + w0_now * Py

    g = np.clip(phi * (1.0 - phi), 0.0, None)
    Lagrange = mu.sum() / g.sum()
    phi = phi + dt * (-divergence_of_flux(phi, ux, uy) - M_phi * (mu - Lagrange * g))

    dvx_dx, dvx_dy = d_dx(vx), d_dy(vx)
    dvy_dx, dvy_dy = d_dx(vy), d_dy(vy)
    Dxx, Dyy = dvx_dx, dvy_dy
    Dxy = 0.5 * (dvx_dy + dvy_dx)
    omega = 0.5 * (dvy_dx - dvx_dy)

    adv_Px = ux * d_dx(Px) + uy * d_dy(Px)
    adv_Py = ux * d_dx(Py) + uy * d_dy(Py)

    Px_new = Px + dt * (-adv_Px - omega * Py + chi_run * (Dxx * Px + Dxy * Py) - hx / Gamma)
    Py_new = Py + dt * (-adv_Py + omega * Px + chi_run * (Dxy * Px + Dyy * Py) - hy / Gamma)

    return phi, Px_new, Py_new, vx, vy


In [ ]:
def run_and_tumble(move=dict(w0=w0, zeta=zeta, beta=beta),
                    xi=xi, chi=chi, kappa=kappa,
                    distance_range=(20.0, 45.0),
                    relax_time=40.0, relax_time_cap=120.0,
                    relax_circularity_target=0.97,
                    n_cycles=4, dt=dt, sample_every=sample_every,
                    frame_every=500, seed=11, R0=R0, eps=eps,
                    max_steps_per_phase=40000, verbose=True):
    """Decide a direction, deform and crawl that way for a (random, by
    default) distance, switch activity off and relax back towards a
    circle, then repeat for `n_cycles`.

    `distance_range`: (min, max) for the random travel distance each
    cycle; pass (d, d) for a fixed distance instead of a random one.
    """
    global zeta_run, beta_run, xi_run, chi_run, kappa_run
    xi_run, chi_run, kappa_run = xi, chi, kappa

    rng = np.random.default_rng(seed)
    phi, Px, Py = initialize(R0, eps, ecc=0.0, P_init="zero", noise=0.0, seed=seed)
    vx, vy = np.zeros_like(phi), np.zeros_like(phi)

    shift_x, shift_y = 0.0, 0.0
    t, step = 0.0, 0

    diagnostics = {"time": [], "x_cm": [], "y_cm": [], "phase": [],
                    "theta_deg": [], "aspect": [], "circularity": [], "v_max": []}
    frames = []

    def recentre():
        nonlocal phi, Px, Py, shift_x, shift_y
        xc, yc = centre_of_mass(phi)
        ix, iy = int(round(-xc / dx)), int(round(-yc / dx))
        if ix or iy:
            phi = np.roll(np.roll(phi, ix, axis=0), iy, axis=1)
            Px = np.roll(np.roll(Px, ix, axis=0), iy, axis=1)
            Py = np.roll(np.roll(Py, ix, axis=0), iy, axis=1)
            shift_x -= ix * dx
            shift_y -= iy * dx
        if touches_boundary(phi):
            print(f"   ** WARNING: cell reached the box edge at t = {t:.0f}; enlarge L")

    def record(phase, theta):
        xc, yc = centre_of_mass(phi)
        x_lab, y_lab = shift_x + xc, shift_y + yc
        d = calculate_diagnostics(phi, Px, Py, vx, vy, Px**2 + Py**2)
        diagnostics["time"].append(t)
        diagnostics["x_cm"].append(x_lab)
        diagnostics["y_cm"].append(y_lab)
        diagnostics["phase"].append(phase)
        diagnostics["theta_deg"].append(np.degrees(theta) if theta is not None else np.nan)
        diagnostics["aspect"].append(d["aspect"])
        diagnostics["circularity"].append(d["circularity"])
        diagnostics["v_max"].append(d["v_max"])
        return x_lab, y_lab

    def diverged():
        return not (np.isfinite(phi).all() and np.isfinite(Px).all() and np.isfinite(Py).all())

    for cycle in range(n_cycles):

        # ---------------- 1. decide ----------------
        theta = rng.uniform(0.0, 2.0 * np.pi)
        target_distance = rng.uniform(*distance_range)
        if verbose:
            print(f"cycle {cycle + 1}/{n_cycles}:  direction = {np.degrees(theta):6.1f} deg,  "
                  f"target distance = {target_distance:5.1f}")

        # the decision takes effect immediately: point P at theta
        Px = phi * np.cos(theta)
        Py = phi * np.sin(theta)
        zeta_run, beta_run = move["zeta"], move["beta"]
        x0, y0 = record("move-start", theta)

        # ---------------- 2. deform and move ----------------
        phase_step = 0
        while True:
            phi, Px, Py, vx, vy = advance_one_step(phi, Px, Py, vx, vy, move["w0"], dt=dt)
            t += dt
            step += 1
            phase_step += 1
            if diverged():
                print(f"   ** diverged at t = {t:.2f}")
                return phi, Px, Py, diagnostics, frames
            if step % frame_every == 0:
                frames.append((t, phi.copy(), Px.copy(), Py.copy(), "move", theta))
            if step % sample_every == 0:
                recentre()
                x_lab, y_lab = record("move", theta)
                travelled = np.hypot(x_lab - x0, y_lab - y0)
                if travelled >= target_distance or phase_step >= max_steps_per_phase:
                    break
        if verbose:
            print(f"   moved {travelled:.1f} in {phase_step * dt:.1f} time units "
                  f"(aspect -> {diagnostics['aspect'][-1]:.2f})")

        # ---------------- 3. relax ----------------
        zeta_run, beta_run = 0.0, 0.0
        relax_t0 = t
        phase_step = 0
        while True:
            phi, Px, Py, vx, vy = advance_one_step(phi, Px, Py, vx, vy, 0.0, dt=dt)
            t += dt
            step += 1
            phase_step += 1
            if diverged():
                print(f"   ** diverged at t = {t:.2f}")
                return phi, Px, Py, diagnostics, frames
            if step % frame_every == 0:
                frames.append((t, phi.copy(), Px.copy(), Py.copy(), "relax", None))
            if step % sample_every == 0:
                recentre()
                record("relax", None)
                circ = diagnostics["circularity"][-1]
                elapsed = t - relax_t0
                if (elapsed >= relax_time_cap
                        or (elapsed >= relax_time and circ >= relax_circularity_target)
                        or phase_step >= max_steps_per_phase):
                    break
        if verbose:
            print(f"   relaxed for {phase_step * dt:.1f} time units "
                  f"(circularity -> {circ:.3f})")

    diagnostics = {k: (v if k == "phase" else np.array(v)) for k, v in diagnostics.items()}
    return phi, Px, Py, diagnostics, frames


# 3. Run it

**This cell is not executed here** — run it on your own machine. With
`n_cycles=4`, a distance drawn from $U(20,45)$ each leg, and a relax
phase that waits for circularity $\geq0.97$ (capped at $120$ time
units so it can't stall forever), expect on the order of $35$–$45$
thousand steps in total. At the Version 3 cost of about
$2.7\,\mathrm{ms}$/step that is roughly $1.5$–$2$ minutes.


In [ ]:
phi_f, Px_f, Py_f, diag_rt, frames_rt = run_and_tumble(
    move=dict(w0=1.0, zeta=-1.0, beta=0.25),
    distance_range=(20.0, 45.0),
    relax_time=40.0, relax_time_cap=120.0, relax_circularity_target=0.97,
    n_cycles=4, frame_every=500, seed=11)

print(f"\n{len(frames_rt)} frames captured over t = 0..{diag_rt['time'][-1]:.0f}")


# 4. Diagnostics — the path, and the shape oscillating between fan and circle

In [ ]:
move_mask = np.array([p in ("move", "move-start") for p in diag_rt["phase"]])

fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot(diag_rt["x_cm"][move_mask], diag_rt["y_cm"][move_mask], ".", color="tab:blue",
        ms=3, label="move")
ax.plot(diag_rt["x_cm"][~move_mask], diag_rt["y_cm"][~move_mask], ".", color="tab:orange",
        ms=3, label="relax")
ax.plot(diag_rt["x_cm"][0], diag_rt["y_cm"][0], "k*", ms=14, label="start")
ax.set_aspect("equal")
ax.set(xlabel="x", ylabel="y", title="Path: decide, move, relax, repeat")
ax.legend()
plt.tight_layout()
plt.savefig("run_and_tumble_path.png", dpi=140)
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))

ax[0].plot(diag_rt["time"], diag_rt["aspect"], color="tab:green")
for t_edge in diag_rt["time"][np.where(np.diff(move_mask.astype(int)) != 0)[0] + 1]:
    ax[0].axvline(t_edge, color="gray", lw=0.6, ls="--")
ax[0].set(xlabel="t", ylabel="aspect ratio",
          title="Shape: fans out while moving, rounds off while relaxing")

ax[1].plot(diag_rt["time"], diag_rt["circularity"], color="tab:purple")
ax[1].axhline(1.0, color="gray", ls=":", lw=1)
ax[1].set(xlabel="t", ylabel="circularity", title="1 = perfect circle")

plt.tight_layout()
plt.savefig("run_and_tumble_diagnostics.png", dpi=140)
plt.show()


# 5. A video of the whole decide–move–relax cycle

Each frame's title shows the current phase and, during a move leg,
the chosen direction in degrees — the same information driving the
simulation, so you can watch the cell commit to a heading, fan out,
travel, then let go and round back up before the next coin flip.


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 6.4))


def render(i):
    ax.clear()
    t, phi_i, Px_i, Py_i, phase, theta = frames_rt[i]
    if phase == "move":
        title = f"t = {t:5.1f}   MOVE   (direction {np.degrees(theta):.0f} deg)"
    else:
        title = f"t = {t:5.1f}   RELAX  (recovering original shape)"
    draw_cell(ax, phi_i, Px_i, Py_i, title=title)


render(0)
fig.tight_layout()


def update(i):
    render(i)
    return []


ani = animation.FuncAnimation(fig, update, frames=len(frames_rt), blit=False)

fps = 12
video_path = "run_and_tumble.mp4"
ani.save(video_path, writer=animation.FFMpegWriter(fps=fps, bitrate=2400))
plt.close(fig)

print(f"saved {video_path}: {len(frames_rt)} frames at {fps} fps "
      f"= {len(frames_rt) / fps:.1f} s")


In [ ]:
Video(video_path, embed=True, width=480)


# 6. Notes

- **What "decide" means here.** There is no deliberation — it is a
  uniform random draw over $[0,2\pi)$ for direction and over
  `distance_range` for how far to go. That is a deliberate
  simplification; a more biologically grounded version would draw the
  *new* direction from something correlated with the *old* one (a
  persistent random walk, as real crawling cells show) rather than a
  fresh, independent coin flip every cycle.
- **Why relax by removing $\zeta,\beta$ rather than reversing them.**
  Switching the active stress and anchoring off (not negative) lets
  the passive physics validated in Version 3 — the same interfacial
  relaxation that pulls an ellipse back to a circle in the passive
  test — do the rounding-off on its own. No new force was added for
  this; it is the existing surface-tension term with nothing left to
  fight it.
- **Fixed vs. random distance.** Pass `distance_range=(d, d)` to
  `run_and_tumble` for a fixed travel distance every cycle instead of
  a random one.
- **Cost.** Four cycles at these settings is on the order of $1.5$–$2$
  minutes on one core (see Section 3). Shorten `n_cycles`,
  `relax_time_cap`, or `distance_range` to iterate faster.
